# Tutorial 07 — Visualization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/tutorials/07_visualization.ipynb)

**Numbers tell the story, but charts sell it.** Vectrix's visualization module (`vectrix.viz`) provides publication-quality interactive charts built on Plotly — designed to work seamlessly with every Vectrix result object.

All charts follow a consistent pattern: pass a Vectrix result, get back a Plotly `go.Figure` you can display inline, save to HTML, or embed in dashboards.

| What you'll learn | Time |
|:------------------|:-----|
| Forecast chart with confidence intervals | 2 min |
| DNA radar chart | 2 min |
| Model comparison heatmap | 2 min |
| Scenario analysis chart | 2 min |
| Backtest performance chart | 2 min |
| Business metrics scorecard | 2 min |
| Composite reports | 2 min |
| Theme customization | 2 min |

In [ ]:
!pip install -q "vectrix[viz]"

## 1. Forecast Chart

The most common visualization — plot predictions with confidence intervals, optionally overlaying historical data for context.

In [ ]:
from vectrix import forecast, loadSample
from vectrix.viz import forecastChart

df = loadSample("airline")
result = forecast(df, steps=24)

fig = forecastChart(result, historical=df)
fig.show()

The chart automatically detects date and value columns from the historical DataFrame. The title defaults to `"Forecast — {model} (MAPE {mape}%)"`.

You can pass a custom title:

In [ ]:
fig = forecastChart(result, historical=df, title="Airline Passengers — 24 Month Forecast")
fig.show()

### Light Theme

Every chart function accepts `theme="light"` for white-background presentations and printed reports.

In [ ]:
fig = forecastChart(result, historical=df, theme="light")
fig.show()

### Forecast Only (No History)

If you only want to show the forecast period, omit the `historical` parameter.

In [ ]:
fig = forecastChart(result)
fig.show()

## 2. DNA Radar Chart

Visualize a time series' statistical fingerprint — 6 normalized features on a polar chart. Instantly shows whether your data is trend-dominated, seasonal, volatile, or forecastable.

| Axis | Feature | Interpretation |
|------|---------|---------------|
| Trend | `trendStrength` | Linear trend dominance (0=flat, 1=strong) |
| Seasonality | `seasonalStrength` | Seasonal pattern strength |
| Memory | `hurstExponent` | Long-range dependency (>0.5 = persistent) |
| Vol. Clustering | `volatilityClustering` | GARCH-like variance clustering |
| Nonlinear | `nonlinearAutocorr` | Nonlinear autocorrelation strength |
| Forecastability | `forecastability` | Overall predictability (1 - spectral entropy) |

In [ ]:
from vectrix import analyze, loadSample
from vectrix.viz import dnaRadar

df = loadSample("airline")
analysis = analyze(df)

fig = dnaRadar(analysis)
fig.show()

Compare radar shapes across different datasets to understand why some are easier to forecast than others.

In [ ]:
df_retail = loadSample("retail")
analysis_retail = analyze(df_retail)

fig = dnaRadar(analysis_retail, title="DNA — Retail Sales")
fig.show()

## 3. Model Comparison Heatmap

After running `compare()`, visualize which models perform best across different error metrics. The heatmap normalizes each column (min-max) so green = best and red = worst.

In [ ]:
from vectrix import compare, loadSample
from vectrix.viz import modelHeatmap

df = loadSample("airline")
comparison = compare(df, steps=12)

fig = modelHeatmap(comparison, top=8)
fig.show()

Show fewer models with a custom title:

In [ ]:
fig = modelHeatmap(comparison, top=5, title="Top 5 Models — Airline Passengers")
fig.show()

## 4. Scenario Analysis Chart

Visualize what-if scenarios from `WhatIfAnalyzer`. The baseline scenario is drawn solid; alternatives are dashed for easy comparison.

In [ ]:
import numpy as np
from vectrix import forecast
from vectrix.business import WhatIfAnalyzer
from vectrix.viz import scenarioChart

data = np.random.randn(200).cumsum() + 500
result = forecast(data, steps=30)

analyzer = WhatIfAnalyzer()
scenarios = analyzer.analyze(
    result.predictions, data,
    [
        {"name": "Optimistic", "trend_change": 0.1},
        {"name": "Pessimistic", "trend_change": -0.15},
        {"name": "Shock", "shock_at": 10, "shock_magnitude": -0.3, "shock_duration": 5},
    ]
)

fig = scenarioChart(scenarios)
fig.show()

### With Forecast Dates

Pass dates for a proper time axis instead of numeric step indices.

In [ ]:
import pandas as pd

dates = pd.date_range("2026-01-01", periods=30, freq="D")
fig = scenarioChart(scenarios, dates=dates, title="Revenue Scenarios — Q1 2026")
fig.show()

## 5. Backtest Chart

Visualize fold-by-fold performance from `Backtester.run()`. Each fold gets a bar; the best fold is green, worst is red, and a dashed line shows the average.

In [ ]:
from vectrix.business import Backtester
from vectrix.engine.ets import AutoETS
from vectrix.viz import backtestChart
import numpy as np

data = np.random.randn(300).cumsum() + 200

bt = Backtester(nFolds=5, horizon=14, strategy="expanding")
bt_result = bt.run(data, lambda: AutoETS())

fig = backtestChart(bt_result)
fig.show()

Switch to RMSE:

In [ ]:
fig = backtestChart(bt_result, metric="rmse", title="Backtest — RMSE by Fold")
fig.show()

## 6. Business Metrics Scorecard

A four-indicator scorecard for business metrics — Accuracy, Bias, WAPE, and MASE. Colors turn green when a metric meets its threshold, red when it doesn't.

In [ ]:
from vectrix.business import BusinessMetrics
from vectrix.viz import metricsCard
import numpy as np

actual = np.array([100, 120, 110, 130, 140, 125, 135, 150, 145, 155])
predicted = np.array([105, 115, 112, 128, 145, 120, 138, 148, 140, 160])

metrics = BusinessMetrics()
m_result = metrics.calculate(actual, predicted)

fig = metricsCard(m_result)
fig.show()

### Custom Thresholds

Default thresholds: Accuracy >= 95%, |Bias| < 3%, WAPE < 5%, MASE < 1.0. Override any of them:

In [ ]:
fig = metricsCard(m_result, thresholds={
    "accuracy": 90,
    "bias": 5,
    "wape": 8,
    "mase": 1.2,
})
fig.show()

## 7. Composite Reports

For dashboards and presentations, composite reports combine multiple charts into a single figure with subplots.

### Forecast Report

A 2-row layout: forecast chart with confidence intervals (top 75%) and error metric bars (bottom 25%).

In [ ]:
from vectrix import forecast, loadSample
from vectrix.viz import forecastReport

df = loadSample("airline")
result = forecast(df, steps=24)

fig = forecastReport(result, historical=df)
fig.show()

### Analysis Report

A 2x2 layout: DNA radar chart (top-left), feature importance bars (top-right), and difficulty score indicator (bottom).

In [ ]:
from vectrix import analyze, loadSample
from vectrix.viz import analysisReport

df = loadSample("airline")
analysis = analyze(df)

fig = analysisReport(analysis)
fig.show()

## 8. Theme Customization

All charts default to the dark theme (navy background, light text). Switch to light theme for presentations and printed reports.

### Using Brand Colors in Custom Charts

Access Vectrix's color system directly for your own Plotly charts:

In [ ]:
from vectrix.viz import COLORS, LIGHT_COLORS, PALETTE, HEIGHT, applyTheme
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[1, 2, 3, 4, 5], y=[10, 20, 15, 25, 22],
    line=dict(color=COLORS["primary"], width=2),
    name="Primary",
))
fig.add_trace(go.Scatter(
    x=[1, 2, 3, 4, 5], y=[8, 15, 18, 20, 26],
    line=dict(color=COLORS["accent"], width=2),
    name="Accent",
))

fig = applyTheme(fig, title="Custom Chart with Vectrix Theme", height=HEIGHT["chart"])
fig.show()

### Color Reference

| Key | Dark | Light | Usage |
|-----|------|-------|-------|
| `primary` | `#06b6d4` | `#06b6d4` | Main brand color (cyan) |
| `accent` | `#8b5cf6` | `#8b5cf6` | Secondary highlight (purple) |
| `positive` | `#10b981` | `#059669` | Good values, below threshold |
| `negative` | `#ef4444` | `#dc2626` | Bad values, above threshold |
| `warning` | `#f59e0b` | `#d97706` | Caution, averages |
| `muted` | `#94a3b8` | `#64748b` | Historical data, secondary |
| `bgDarker` | `#020617` | `#f8fafc` | Paper background |
| `bg` | `#0f172a` | `#ffffff` | Plot background |
| `card` | `#1e293b` | `#f1f5f9` | Card/panel background |
| `text` | `#f8fafc` | `#0f172a` | Primary text |
| `border` | `#334155` | `#e2e8f0` | Borders, dividers |

### Height Constants

```python
HEIGHT["chart"]     # 480 — individual charts
HEIGHT["card"]      # 240 — metrics cards
HEIGHT["report"]    # 640 — forecast report
HEIGHT["analysis"]  # 680 — analysis report
HEIGHT["small"]     # 360 — compact charts
```

## 9. Saving Charts

Every chart returns a standard Plotly `go.Figure`. Use Plotly's built-in export methods:

In [ ]:
fig = forecastChart(result, historical=df)

fig.write_html("forecast.html")

print("Saved to forecast.html")
print("For PNG export: pip install kaleido, then fig.write_image('forecast.png')")

## 10. Full Dashboard Example

Build a complete analysis dashboard by combining multiple charts in one workflow.

In [ ]:
from vectrix import forecast, analyze, compare, loadSample
from vectrix.viz import forecastReport, analysisReport, modelHeatmap

df = loadSample("airline")

result = forecast(df, steps=24)
analysis = analyze(df)
comparison = compare(df, steps=24)

In [ ]:
fig1 = forecastReport(result, historical=df, title="Airline Forecast Report")
fig1.show()

In [ ]:
fig2 = analysisReport(analysis, title="Airline DNA Analysis")
fig2.show()

In [ ]:
fig3 = modelHeatmap(comparison, top=8, title="Model Comparison — Airline")
fig3.show()

Each figure is independent — display them in separate cells, save to separate HTML files, or embed in a web application.

---

**Previous:** [Tutorial 06 — Business Intelligence](06_business.ipynb)